# Piloto 2026 · recalibración de aromas con rCO₂ liberado

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "fermentation_model").exists():
    ROOT = ROOT.parent
if not (ROOT / "fermentation_model").exists():
    raise RuntimeError("Execute from the repository or a descendant directory")
import os
sys.path.insert(0, str(ROOT / "fermentation_model"))
from pilot_2026 import run_aroma_co2_release_recalibration_2026 as analysis
result = analysis.load_results() if os.environ.get("PILOT_AROMA_REUSE_RESULTS") == "1" else analysis.run_analysis()
print("Resultados:", analysis.RESULTS_DIR.relative_to(ROOT))

## tl;dr

In [ ]:
summary = result["summary"]
display(summary["holdout_wine_rmse_comparison"])
print("Mediana CO2 liberado / estequiométrico integrado:", round(summary["median_released_over_stoichiometric_integral"], 4))
print("Pérdida identificada:", summary["loss_identified_with_released_co2"])
print("Gate:", result["gate"]["verdict"])

## Contexto y métodos

- Se conservan formación aromática, partición UNIFAC, captura, errores y censura del modelo anterior.
- Se sustituye rCO₂ estequiométrico por rCO₂ emitido predicho por `solubility_o2_nitrogen_boost_continuous_release`.
- El forcing usa parámetros CO₂ específicos de lote y la ganancia de matriz validada.
- `26158` y `26211` son holdouts tanto para CO₂ como para aromas.
- La primera muestra de vino inicializa la trayectoria y no entra al RMSE de validación; las muestras siguientes sí.
- Se reestima `effective_loss_scale`: su valor absoluto depende de la escala del forcing rCO₂.

## Datos y forcing rCO₂

In [ ]:
display(result["exposure"].round(4))
display(Image(filename=analysis.FIGURE_DIR / "rco2_forcing_comparison.png"))
display(Image(filename=analysis.FIGURE_DIR / "integrated_rco2_exposure.png"))

## Resultados de calibración y validación

In [ ]:
display(result["parameters"].query("fit_scope == 'all_data'").round(5))
display(result["metrics"].query("fit_scope == 'calibration_only' and role == 'holdout'").round(4))
display(Image(filename=analysis.FIGURE_DIR / "holdout_wine_predictions.png"))
display(Image(filename=analysis.FIGURE_DIR / "holdout_wine_rmse_comparison.png"))

In [ ]:
display(Image(filename=analysis.FIGURE_DIR / "holdout_condensate_predictions.png"))
display(Image(filename=analysis.FIGURE_DIR / "aroma_parameter_comparison.png"))

## Identificabilidad

In [ ]:
display(result["validation"].query("forcing_source == 'released_co2_model' and fit_scope == 'all_data'").round(4))
display(Image(filename=analysis.FIGURE_DIR / "release_forcing_profile_likelihood.png"))

## Takeaways

La comparación relevante es el desempeño holdout después de volver a estimar la escala de pérdida para cada definición de rCO₂. Una mejora del forcing no implica por sí sola identificabilidad completa: `effective_loss_scale` sigue siendo condicional a la curva de CO₂ fijada y a la información disponible en condensados. Los límites de detección se incorporan como censura, no como ceros.

In [ ]:
print("Notebook ejecutado sin errores.")
print("Figuras embebidas:", len(result["figures"]))
print("Artefactos:", analysis.RESULTS_DIR.relative_to(ROOT))